In [1]:
# ------------------ Initial setup ---------------------
import gc
import torch
from huggingface_hub import get_token
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

set_seed(42)
torch.set_num_threads(2)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

# Read a local saved login or HF_TOKEN; never store a token in the notebook.
hf_token = get_token()
model_id = "meta-llama/Llama-3.2-3B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.float16, low_cpu_mem_usage=True, token=hf_token,
).to(device).eval()
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token, clean_up_tokenization_spaces=False)
tokenizer.pad_token = tokenizer.eos_token


def make_a_query(prompt: str, tokenizer, model, device, max_new_tokens: int = 200):
    # Chat-template prompts already contain special tokens; do not add another BOS.
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(device)
    prompt_length = inputs["input_ids"].shape[1]
    eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.15,
            top_p=0.9,
            num_beams=1,
            # Stop on Llama's end-of-turn marker and the model's EOS marker.
            eos_token_id=list(dict.fromkeys([eot_id, tokenizer.eos_token_id])),
            pad_token_id=tokenizer.pad_token_id,
        )
    answer = tokenizer.decode(output[0, prompt_length:], skip_special_tokens=True).strip()
    del output, inputs
    gc.collect()
    if getattr(device, "type", str(device)) == "mps":
        torch.mps.empty_cache()
    return answer


/Users/nicolabeccaceci/.venvs/llms-lab07/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: mps


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:  50%|█████     | 1/2 [00:30<00:30, 30.54s/it]

Loading checkpoint shards: 100%|██████████| 2/2 [00:36<00:00, 16.19s/it]

Loading checkpoint shards: 100%|██████████| 2/2 [00:36<00:00, 18.34s/it]

# Exercise: User Story generation

User stories are brief, simple descriptions of a feature or requirement from the perspective of an end user. They are used in Agile software development to capture the user’s needs and goals in a way that prioritizes functionality and value. A typical user story follows the format:

"As a [type of user], I want [a specific goal] so that [reason or benefit]."

This format ensures the focus stays on delivering solutions that meet the user’s expectations. User stories guide development by defining clear, actionable tasks for teams while remaining flexible and open to adaptation as the project evolves.

Our objective is to create an agent architecture like the following one:

1. **Problem Specification (Domain Specification):**
The process begins with the definition of a problem description, which outlines the domain or context of the system. This serves as the input to the LLM and sets the foundation for generating user stories. Our problem specification is provided as the input to the process.

2. **Role(s) Extraction ("Who" Aspect):**
A prompt is used to extract roles from the problem description.
Roles represent the different types of users or stakeholders involved in the system This step answers "who" the user stories are about.

3. **Function(s) Extraction ("What" Aspect):**
Another prompt is used to extract functions associated with each role.
Functions describe the specific actions or goals the roles aim to perform within the system. This step answers "what" the users want to achieve.

4. **Purpose(s) Extraction ("Why" Aspect):**
A subsequent prompt is used to determine the purposes behind the functions identified. Purposes explain the reasons or motivations behind the desired actions of the roles. This step answers "why" the users need the functions.

5. **User Stories Generation:**
Combining the roles (who), functions (what), and purposes (why), a final prompt generates complete user stories. These user stories take the standard Agile format: "As a \<who\>, I want \<what\>, so that \<why\>."

**Prompt Chain**:
The process is iterative, as each step informs the next and ensures that the roles, functions, and purposes are fully aligned. The LLM is guided by a structured "prompt chain," where prompts are carefully designed to extract specific information at each stage.

**Output**:
The final output consists of user stories that capture the needs and goals of the system's users in a concise, actionable format. These stories guide development teams in implementing the desired features and functionalities.






In [2]:
#reference requirements_text

requirements_text = "The proposed platform is designed to enhance the hiking experience for various user groups, including visitors, local guides, platform managers, and hut workers. The platform provides a centralized repository of hiking routes, hut information, and parking facilities. It also enables interactive features such as real-time hike tracking, personalized recommendations, and group hike planning. By combining these capabilities, the platform seeks to foster safe, informed, and collaborative hiking experiences.\
The platform will be deployed as a cloud-based web and mobile application accessible to all stakeholders. The distribution strategy includes an app available on major mobile operating systems, such as iOS and Android, alongside a responsive web interface. It will require an internet connection for features like real-time tracking, notifications, and user authentication, though some offline capabilities, such as pre-downloaded hike information, will also be available.\
User authentication will be role-based, ensuring that only authorized users, such as verified hut workers and platform managers, can access sensitive or administrative features.\
Visitors are the primary users of the platform. They can browse a comprehensive list of hiking trails, filter them based on specific criteria such as difficulty, length, or starting point, and view detailed descriptions. To access advanced features like personalized recommendations, visitors can create user accounts by registering on the platform. Registered users can record their fitness parameters, enabling the system to suggest trails tailored to their capabilities.\
During a hike, visitors can record their progress by marking reference points and sharing their live location through a broadcasting URL. They can also initiate group activities by planning hikes, adding group members, and confirming group participation. The platform allows visitors to start, terminate, and track their hikes, with notifications for unfinished hikes or late group members to ensure safety and accountability.\
Local guides enrich the platform by contributing essential information. They can add detailed descriptions of hikes, parking facilities, and huts, ensuring hikers have accurate and comprehensive data. Local guides also link parking lots and huts to specific trails as starting or arrival points, enhancing the planning process.\
To aid in the visual representation and accessibility of information, local guides can upload pictures of huts and connect these locations directly to hikes. This integration simplifies route planning and helps visitors visualize their journey.\
Platform managers oversee the operational integrity and safety of the platform. They verify new hut worker registrations, ensuring that only authorized personnel can update hut-related data. Managers can also broadcast weather alerts for specific areas, notifying all hikers in those regions through push notifications. This ensures that users stay informed about potentially hazardous conditions.\
The platform manager's role includes maintaining an organized and secure user system while facilitating collaboration between local guides, hut workers, and visitors.\
Hut workers are critical to the maintenance of up-to-date trail and accommodation information. After registering and being verified, hut workers can log into the platform to add or update information about their assigned huts, including uploading pictures and describing the facilities available. They can also monitor and report on the condition of nearby trails, ensuring hikers receive current information.\
Hut workers play a vital role in providing situational updates for hikers. For instance, if a nearby trail is impacted by severe weather or physical obstructions, they can communicate these conditions through the platform. This enhances the safety and preparedness of all hikers relying on the platform."


#---------------------Example usage - without chat template and customized system prompt------------------
prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert on world capitals.
Respond with only the capital city of the given country. Do not repeat the question.<|eot_id|>
<|start_header_id|>user<|end_header_id|>

What is the capital of France?<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>

"""

response = make_a_query(prompt, tokenizer, model, device)

print(f"Prompt: {prompt}\n\nResponse: {response}")
#----------------------------------------------------------------------------


#---------------------Example usage - with chat template and customized system prompt---------------------
messages = [
    {"role": "system", "content": "You are an expert on world capitals. Respond with only the capital city of the given country. Do not repeat the question."},
    {"role": "user", "content": "What is the capital of France?"}
]

chat = tokenizer.apply_chat_template(messages,
                                     tokenize=False, #IMPORTANT: only the special tokens are added, we'll apply tokenisation later
                                     add_generation_prompt=True) # adds the placeholder for the assistant's response


response = make_a_query(chat, tokenizer, model, device)

print("------------------------------------------------------------")
print(f"Prompt: {chat}\n\nResponse: {response}")

Prompt: <|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert on world capitals.
Respond with only the capital city of the given country. Do not repeat the question.<|eot_id|>
<|start_header_id|>user<|end_header_id|>

What is the capital of France?<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>



Response: Paris


------------------------------------------------------------
Prompt: <|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Oct 2026

You are an expert on world capitals. Respond with only the capital city of the given country. Do not repeat the question.<|eot_id|><|start_header_id|>user<|end_header_id|>

What is the capital of France?<|eot_id|><|start_header_id|>assistant<|end_header_id|>



Response: Paris


## Extracting the Actors

To automate the extraction of roles ("Who" aspect) from a problem description, we utilize a language model (LLM) to identify key actors or stakeholders involved in the system. Roles, such as "visitors," "local guides," "platform managers," and "hut workers," represent the primary users or contributors to the system and are essential for generating accurate and actionable user stories. This step focuses on using the LLM to extract these roles and then validating the quality of the extraction by comparing the results against a reference dataset.

The LLM is provided with a detailed problem description and prompted to extract the roles (actors) associated with the system. The prompt is carefully designed to ensure the LLM captures all relevant **roles** based on the context of the problem. The output is a list of roles that the LLM determines as key stakeholders in the system.







In [3]:
#define three different prompts for extracting users, using the zero-shot, one-shot and few-shot approach. then evaluate the difference in terms of capability to format the output and actual results

# IMPORTANT: wheter you use the chat template or not, remember to add the shots into the correct structure composed by user's query and assistant's response (see previous lab)

query = "Requirements Text:\n" + requirements_text + "\nList the distinct user roles."

shots = [ # This is a list of lists, useful to separate each shot from each other

    # First shot
    [
        {'role': 'user',
          'content': """Requirements Text:
"The application is designed to streamline food delivery operations. Customers can browse menus and place orders. Delivery personnel use the app to view delivery assignments and update order statuses. Restaurant managers can manage their menus and track incoming orders."""},
        {'role': 'assistant', 'content': """1. Customers
2. Delivery Personnel
3. Restaurant Managers"""},
    ],

    # Second shot
    [
      {'role': 'user',
      'content': """Requirements Text:
The platform supports the education sector by providing tools for students, teachers, and administrators. Students can access course materials, submit assignments, and track their progress. Teachers can create content, grade submissions, and communicate with students. Administrators oversee user registrations and system performance."""},
      {'role': 'assistant',
      'content': """1. Students
2. Teachers
3. Administrators"""}
    ],

    # Third shot
    [
        {'role': 'user',
         'content': """Requirements Text:
The system provides real-time updates for commuters, transit operators, and city planners. Commuters can check schedules and delays. Transit operators monitor vehicle statuses and manage timetables. City planners analyze system data to improve routes and infrastructure."""},
        {'role': 'assistant',
         'content': """1. Commuters
2. Transit Operators
3. City Planners"""}
    ]
]


system_prompt = """Extract only the distinct user roles explicitly supported by the supplied requirements.
Return only the role names explicitly stated in the requirements, one per numbered line in the form `1. Role name`.
Do not include a heading, introduction, rationale, examples, actions, or commentary.
Stop immediately after the final role. Do not infer roles absent from the text."""

# Build the message list for each promoting technique

messages_zero_shot = [{"role": "system", "content": system_prompt}, {"role": "user", "content": query}]
prompt_zero_shot = tokenizer.apply_chat_template(messages_zero_shot, tokenize=False, add_generation_prompt=True)

messages_one_shot = [{"role": "system", "content": system_prompt}] + shots[0] + [{"role": "user", "content": query}]
prompt_one_shot = tokenizer.apply_chat_template(messages_one_shot, tokenize=False, add_generation_prompt=True)

messages_few_shots = [{"role": "system", "content": system_prompt}] + [message for shot in shots for message in shot] + [{"role": "user", "content": query}]
prompt_few_shots = tokenizer.apply_chat_template(messages_few_shots, tokenize=False, add_generation_prompt=True)


# Call the LLM function for the three prompts and print the results
roles_extracted_zero_shot = make_a_query(prompt_zero_shot, tokenizer, model, device, max_new_tokens=96)

print("Extracted Roles - zero shot: \n", roles_extracted_zero_shot)

roles_extracted_one_shot = make_a_query(prompt_one_shot, tokenizer, model, device, max_new_tokens=96)

print("Extracted Roles - one shot: \n", roles_extracted_one_shot)


roles_extracted_few_shots = make_a_query(prompt_few_shots, tokenizer, model, device, max_new_tokens=96)

print("Extracted Roles - few shots: \n", roles_extracted_few_shots)

Extracted Roles - zero shot: 
 1. Visitors
2. Local Guides
3. Platform Managers
4. Hut Workers


Extracted Roles - one shot: 
 1. Visitors
2. Local Guides
3. Platform Managers
4. Hut Workers


Extracted Roles - few shots: 
 1. Visitors
2. Local Guides
3. Platform Managers
4. Hut Workers


While no external dataset is used to refine or influence the role extraction process, we validate the extracted roles against a reference dataset contained in a CSV file named stories.csv. This file includes a specific column listing all the predefined roles associated with the system. These roles serve as the ground truth for validation purposes.


To assess the accuracy and completeness of the LLM’s role extraction, we compute the precision and recall metrics by comparing the LLM's output to the ground truth roles:

- **Precision**: Measures how many of the roles extracted by the LLM are correct (i.e., exist in the reference dataset).

  $P = \frac{True\ Positives}{True\ Positives + False\ Positives}$


- **Recall**: Measures how many of the ground truth roles were successfully identified by the LLM.

  $R = \frac{True\ Positives}{True\ Positives + False\ Negatives}$


After extracting the roles using the LLM, the results are compared to the roles in the reference column from the CSV file. For each role identified by the LLM:

- If the role exists in the reference dataset, it is considered a true positive.
- If the role does not exist in the reference dataset, it is marked as a false positive.
- Any role in the reference dataset that the LLM fails to identify is considered a false negative.

The final output of this process includes:
- A list of roles extracted by the LLM.
- Precision and recall metrics, which provide a quantitative assessment of the LLM’s performance in role extraction.

In [4]:
import csv
import re


def extract_unique_roles(file_path: str):
    roles = set()
    with open(file_path, newline="", encoding="utf-8-sig") as stream:
        for row in csv.DictReader(stream):
            match = re.search(r"^As\s+(?:an?\s+)?(.+?)\s+I\s+want\b", row["Description"].strip(), flags=re.IGNORECASE)
            if match:
                roles.add(match.group(1).strip().lower())
    return sorted(roles)


def extract_list_items(text: str):
    """Keep only a leading numbered/bulleted list; reject explanatory prose."""
    items = []
    started = False
    for line in text.splitlines():
        line = line.strip()
        match = re.match(r"^(?:\d+\s*[.)-]?\s+|[-*•]\s+)(.+?)\s*$", line)
        if match:
            started = True
            item = match.group(1).strip().strip("* ")
            # Keep the role label and discard parenthetical/editorial commentary.
            item = re.split(r"\s+[–—]\s+|\s+-\s+", item, maxsplit=1)[0]
            item = re.sub(r"\s*\([^)]*\)", "", item).strip()
            item = re.sub(r"/s$", "s", item, flags=re.IGNORECASE)
            if item:
                items.append(item)
        elif started and line:
            break
    return list(dict.fromkeys(items))


def normalize_roles(role_list: list[str]):
    # Ignore whitespace, case and a final plural s as requested by the exercise.
    return {re.sub(r"\s+", "", role.lower()).removesuffix("s") for role in role_list if role.strip()}


def role_metrics(expected, actual):
    expected, actual = normalize_roles(expected), normalize_roles(actual)
    tp = len(expected & actual)
    fp = len(actual - expected)
    fn = len(expected - actual)
    return tp, fp, fn, tp / (tp + fp) if tp + fp else 0.0, tp / (tp + fn) if tp + fn else 0.0


file_path = "stories.csv"
expected_roles = extract_unique_roles(file_path)
print("Reference roles from stories.csv:", expected_roles)

Reference roles from stories.csv: ['friend (of a hiker)', 'hiker', 'hut worker', 'local guide', 'platform manager', 'visitor']


In [5]:
actual_roles = extract_list_items(roles_extracted_few_shots)
print("Roles extracted by the model:", actual_roles)
true_positives, false_positives, false_negatives, precision, recall = role_metrics(expected_roles, actual_roles)
print("True Positives:", true_positives)
print("False Positives:", false_positives)
print("False Negatives:", false_negatives)
print("Precision:", precision)
print("Recall:", recall)


Roles extracted by the model: ['Visitors', 'Local Guides', 'Platform Managers', 'Hut Workers']
True Positives: 4
False Positives: 0
False Negatives: 2
Precision: 1.0
Recall: 0.6666666666666666


## Extracting the Functions

In this step, the primary goal is to automate the identification of functions required by the roles identified in the previous step (Role Extraction). The function refers to the tasks or actions that the roles need to perform within the system. To achieve this, a language model (LLM) is used to analyze the problem description, and based on the roles identified, it is prompted to extract the functions those roles desire to perform.

For example, after identifying the role of "visitors" or "local guides," the LLM is prompted to determine the specific actions these roles want to carry out within the system, such as "view hikes," "add descriptions," or "manage registrations." This extraction process is critical for understanding the functionality needed in the system to meet the requirements of its stakeholders.

By leveraging the context and the roles passed from the previous step, the LLM extracts a list of desired functions. This list helps define the operational scope of the system and lays the foundation for generating user stories in subsequent steps. The extracted functions are validated to ensure completeness and alignment with the roles' needs.


In [6]:
current_roles = actual_roles
user_stories = list()

#this function will be used to extract the functions from the answer provided by the llm agent

def extract_functions(text: str):
    """Accept only an initial, contiguous numbered/bulleted list of functions."""
    functions = []
    started = False
    for line in text.splitlines():
        line = line.strip()
        if not line:
            if started:
                break
            continue
        if line.upper() == "NONE":
            return []
        match = re.match(r"^(?:\d+[.)]|[-*•])\s+(.+?)\s*$", line)
        if not match:
            if started:
                break
            return []
        started = True
        item = match.group(1).strip().rstrip(".")
        if item:
            functions.append(item)
    return list(dict.fromkeys(functions))



#this function can be used to format a user story in the expected format. it returns a list of tuples (user, story)

def create_user_stories(role: str, functions: list[str]):
    """
    Generate user stories in the standard format.
    """
    user_stories = [(role, f"As a {role}, I want to {function}") for function in functions]
    return user_stories


for role in current_roles:

    #1. Loop over the list of roles and extract the piece of text from requirements related to that role
    print("\n\nExtracting stories for role", role)


    #define a prompt to extract functions for a given user
    #the prompt will be composed of the following fields:
    # - context: usage persona of the Lllama agent and instructions
    # - examples: examples of requirements text, role and user
    # - task: description of the task to perform. here you have to provide the current role
    # - answer field: placeholder from which the response will start

    #a tentative example can be the following one (you can generate many of them for instance by using another LLM agent, e.g. ChatGPT)
    example_text = [
        {
            'role': 'user',
            'content': """**Requirements Text**:
The application is designed to streamline food delivery operations. Customers can browse menus and place orders. Delivery personnel use the app to view delivery assignments and update order statuses. Restaurant managers can manage their menus and track incoming orders.

Role: Delivery personnel
List the functions for this role."""
        },

        {
            'role': 'assistant',
            'content': """1. View delivery assignments.
2. Update order status."""
        }
    ]

    system_prompt = (
        "Extract only actions explicitly supported by the requirements for the requested role. "
        "Return 1 to 8 numbered lines, each formatted `1. Verb phrase` and no more than eight words. "
        "Use no heading, explanation, prose, claims, or action not stated in the requirements. "
        "Stop after the final list item. If no action is stated for that role, return exactly `NONE`."
    )

    query = f"Requirements Text:\n{requirements_text}\n\nRole: {role}\nList the functions for this role."

    messages = [{"role": "system", "content": system_prompt}] + example_text + [{"role": "user", "content": query}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


    #now ask the agent and get the response
    response = make_a_query(prompt, tokenizer, model, device, max_new_tokens=128)  # Memory-safe cap for a concise numbered action list

    print(response)

    functions = extract_functions(response)

    tmp_user_stories = create_user_stories(role, functions)
    if not functions:
        if response.strip().upper() == "NONE":
            print("No explicit supported functions were returned for this role.")
            continue
        raise ValueError(f"The model returned no numbered functions for {role}: {response}")
    user_stories.extend(tmp_user_stories)



Extracting stories for role Visitors


1. Browse hiking trails.
2. Filter hiking trails.
3. View detailed trail descriptions.
4. Record fitness parameters.
5. Record hike progress.
6. Share live location.
7. Plan group hikes.
8. Track hikes.


Extracting stories for role Local Guides


1. Add detailed hike descriptions.
2. Link parking lots to trails.
3. Upload pictures of huts.
4. Connect huts to hikes.


Extracting stories for role Platform Managers


1. Verify new hut worker registrations.
2. Broadcast weather alerts.
3. Maintain an organized user system.
4. Facilitate collaboration between local guides, hut workers, and visitors.


Extracting stories for role Hut Workers


1. Add or update hut information.
2. Upload pictures of huts.
3. Describe hut facilities.
4. Monitor and report trail conditions.
5. Communicate situational updates to hikers.


## Extracting the purposes

In this step, the focus is on extracting the purposes behind the functions identified in the previous step. The purpose refers to the underlying reasons or motivations that drive a role to perform a particular function within the system. The language model (LLM) is prompted to identify these reasons by analyzing the problem description and the functions extracted in the previous step.

After the functions (what the roles want to do) are identified, the LLM is tasked with understanding and extracting the why behind each function. For example, a role like "hiker" may want to "register for a hike" (function), but the purpose behind this action might be "so that they can track their progress" or "to ensure they can participate in the hike." Similarly, a "local guide" may want to "add a hike description," and the purpose could be "to provide helpful information to other users."

The extracted purposes help provide a deeper understanding of why each function is important to the roles. These purposes are essential for refining the system’s objectives and ensuring that the design aligns with the actual needs and motivations of the stakeholders. The LLM identifies these purposes by analyzing both the context of the roles and the corresponding functions, ensuring that all relevant purposes are captured to provide clarity for the user stories that will follow in the next step.

In [7]:
# Demonstrate purpose extraction for one (role, story) pair, as requested.
if not user_stories:
    raise ValueError("No user stories were generated in the previous step.")
user_story1 = user_stories[0]
role, initial_story = user_story1
prefix = f"As a {role}, I want to "
function = initial_story[len(prefix):] if initial_story.startswith(prefix) else initial_story
print("Role:", role)
print("Function:", function)

system_prompt = (
    "You are a requirements analyst. Use the requirements and the given role and function "
    "to infer the supported purpose. Return exactly one complete story in the format "
    "'As a <role>, I want to <function>, so that <purpose>.' Do not invent capabilities "
    "or add an explanation."
)
shot_examples = [
    {"role": "user", "content": "Requirements: Visitors filter hiking trails by difficulty, length, or starting point to select a suitable route.\nRole: visitor\nFunction: filter hiking trails by difficulty"},
    {"role": "assistant", "content": "As a visitor, I want to filter hiking trails by difficulty, so that I can choose a route suited to my abilities."},
]
query = f"Requirements:\n{requirements_text}\n\nRole: {role}\nFunction: {function}"
messages = [{"role": "system", "content": system_prompt}] + shot_examples + [{"role": "user", "content": query}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
response = make_a_query(prompt, tokenizer, model, device, max_new_tokens=128)  # One concise user-story purpose clause
print(response)

Role: Visitors
Function: Browse hiking trails


As a visitor, I want to browse hiking trails, so that I can explore and discover suitable routes for my hiking experience.
